In [4]:
%pip install duckdb -q


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:

import pandas as pd, pyarrow.parquet as pq, pyarrow as pa
import pyarrow.compute as pc, numpy as np, glob, os, gc, re
try:
    import duckdb
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb

SAVE_DIR = '/srv/work'
RAW_PATH = f'{SAVE_DIR}/te_all_raw_v4.parquet'

COLS = ['_time', 'source_agent', 'target_agent', 'source_loc', 'target_loc',
        'cloud_provider', 'region', 'metric', 'value', 'test_name', 'test_type']

METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']
METRICS_BGP = ['bgp.path_changes.count', 'bgp.updates.count', 'bgp.reachability']

ARCHIVOS = sorted(
    glob.glob(f'{SAVE_DIR}/te_*_raw_may_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_w[0-9].parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_apr_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_jul_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_aug_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_sep_w*.parquet')
)

TE_CONFIG = {
    'interval_s': 300, 'protocol': 'TCP', 'port': 49153,
    'dscp': 'EF (46)', 'direction': 'bidirectional', 'latency_type': 'one-way',
    'bgp_enabled': True, 'bgp_public': False,
    'n_agents_configured': 46, 'n_destinations': 24,
    'n_bgp_monitors_latam': 12, 'n_source_countries': 12,
}

print(f"Config TE: {TE_CONFIG['interval_s']}s · {TE_CONFIG['protocol']}/{TE_CONFIG['port']}"
      f" · DSCP={TE_CONFIG['dscp']} · latencia={TE_CONFIG['latency_type']}")
print(f"Agentes: {TE_CONFIG['n_agents_configured']} conf · {TE_CONFIG['n_source_countries']} países"
      f" | Destinos: {TE_CONFIG['n_destinations']} | BGP public={TE_CONFIG['bgp_public']}")
print(f"\nParquets encontrados : {len(ARCHIVOS)}")
if len(ARCHIVOS) == 72:
    print("  ✅ 72 parquets confirmados (39 originales + 33 nuevos)")
else:
    print(f"  ⚠️  ADVERTENCIA: se esperaban 72, encontrados {len(ARCHIVOS)}")
    for f in ARCHIVOS:
        print(f"    {os.path.basename(f)}")

raw_exists = os.path.exists(RAW_PATH)
if raw_exists:
    size_mb = os.path.getsize(RAW_PATH) / 1e6
    print(f"\n✅ te_all_raw_v4.parquet existe: {size_mb:.0f} MB — celda 1 omite consolidación")
else:
    print(f"\n⚠️  te_all_raw_v4.parquet NO existe — ejecutar celda 1")

print("\nv10 setup completo")


Config TE: 300s · TCP/49153 · DSCP=EF (46) · latencia=one-way
Agentes: 46 conf · 12 países | Destinos: 24 | BGP public=False

Parquets encontrados : 72
  ✅ 72 parquets confirmados (39 originales + 33 nuevos)

✅ te_all_raw_v4.parquet existe: 3306 MB — celda 1 omite consolidación

v10 setup completo


In [2]:

if os.path.exists(RAW_PATH):
    size_mb = os.path.getsize(RAW_PATH) / 1e6
    meta = pq.read_metadata(RAW_PATH)
    print(f"✅ te_all_raw_v4.parquet ya existe")
    print(f"   Tamaño   : {size_mb:.0f} MB")
    print(f"   Row groups: {meta.num_row_groups}")
    print(f"   Filas     : {meta.num_rows:,}")
    print("   Celda 1 omitida — continuando con celda 2")
else:
    print(f"Consolidando {len(ARCHIVOS)} parquets → {RAW_PATH}")
    print("Esto puede tardar ~20-30 minutos. Pico RAM estimado: < 2 GB")
    print("─" * 60)

    faltantes = [f for f in ARCHIVOS if not os.path.exists(f)]
    if faltantes:
        raise FileNotFoundError(
            f"Faltan {len(faltantes)} parquets antes de consolidar:\n"
            + "\n".join(f"  {f}" for f in faltantes)
        )

    writer = None
    total_rows = 0

    for i, f in enumerate(ARCHIVOS):
        table = pq.read_table(f).select(COLS)
        n = len(table)
        total_rows += n

        if writer is None:
            writer = pq.ParquetWriter(RAW_PATH, table.schema, compression='snappy')
        writer.write_table(table)

        if (i + 1) % 10 == 0 or i == len(ARCHIVOS) - 1:
            pct = (i + 1) / len(ARCHIVOS) * 100
            print(f"  [{i+1:2d}/{len(ARCHIVOS)}] {os.path.basename(f):<45} "
                  f"filas={n:>9,}  total={total_rows:>12,}  ({pct:.0f}%)")

        del table; gc.collect()

    if writer:
        writer.close()

    size_mb = os.path.getsize(RAW_PATH) / 1e6
    print(f"\n✅ Consolidación completa")
    print(f"   Filas totales : {total_rows:,}")
    print(f"   Tamaño final  : {size_mb:.0f} MB")
    print(f"   Destino       : {RAW_PATH}")


Task was destroyed but it is pending!
task: <Task pending name='Task-32' coro=<_async_in_context.<locals>.run_in_context() done, defined at /usr/local/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-33' coro=<Kernel.shell_main() running at /usr/local/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /usr/local/lib64/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/local/lib64/python3.11/site-packages/pyarrow/parquet/core.py:1393: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  import pyarrow.dataset as ds
Task was destroyed but it is pending!
task: <Task pending name='Task-33' coro=<Kernel.shell_main() running at /usr/local/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


Consolidando 72 parquets → /srv/work/te_all_raw_v4.parquet
Esto puede tardar ~20-30 minutos. Pico RAM estimado: < 2 GB
────────────────────────────────────────────────────────────
  [10/72] te_aws_raw_may_w3.parquet                     filas=8,425,486  total=  87,736,078  (14%)
  [20/72] te_aws_raw_w4.parquet                         filas=8,566,592  total= 176,733,971  (28%)
  [30/72] te_azure_raw_aug_w5.parquet                   filas=5,169,518  total= 246,107,969  (42%)
  [40/72] te_azure_raw_sep_w4.parquet                   filas=5,934,541  total= 315,372,512  (56%)
  [50/72] te_gcp_raw_aug_w1.parquet                     filas=6,953,353  total= 379,519,419  (69%)
  [60/72] te_gcp_raw_may_w5.parquet                     filas=3,606,176  total= 443,055,567  (83%)
  [70/72] te_gcp_raw_w6.parquet                         filas=6,290,430  total= 511,767,981  (97%)
  [72/72] te_gcp_raw_w8.parquet                         filas=6,288,336  total= 525,019,966  (100%)

✅ Consolidación completa
 

In [3]:

import collections

def epoch_label(fname):
    base = os.path.basename(fname)
    for tok in ['apr', 'may', 'jul', 'aug', 'sep']:
        if f'_raw_{tok}' in base or (tok == 'may' and '_raw_may' in base):
            return tok
    if '_raw_w' in base:
        return 'jun-jul'
    return 'otros'

print("=" * 70)
print("INVENTARIO DE ARCHIVOS RAW — 72 parquets")
print("=" * 70)

by_cloud  = collections.defaultdict(lambda: {'n': 0, 'mb': 0})
by_epoch  = collections.defaultdict(lambda: {'n': 0, 'mb': 0})
total_mb  = 0

for f in ARCHIVOS:
    mb = os.path.getsize(f) / 1e6
    total_mb += mb
    base = os.path.basename(f)
    m = re.match(r'te_([^_]+)_', base)
    cloud = m.group(1).upper() if m else '?'
    ep = epoch_label(f)
    by_cloud[cloud]['n']  += 1
    by_cloud[cloud]['mb'] += mb
    by_epoch[ep]['n']     += 1
    by_epoch[ep]['mb']    += mb

print(f"\nPOR PROVEEDOR CLOUD:")
print(f"  {'Cloud':<8} {'Archivos':>8}  {'MB':>8}  {'MB/arch':>8}")
print(f"  {'-'*36}")
for cloud, v in sorted(by_cloud.items()):
    avg = v['mb'] / v['n'] if v['n'] else 0
    print(f"  {cloud:<8} {v['n']:>8}  {v['mb']:>8.0f}  {avg:>8.1f}")

print(f"\nPOR ÉPOCA TEMPORAL:")
print(f"  {'Época':<10} {'Archivos':>8}  {'MB':>8}")
print(f"  {'-'*30}")
for ep, v in sorted(by_epoch.items()):
    print(f"  {ep:<10} {v['n']:>8}  {v['mb']:>8.0f}")

print(f"\nTOTAL  archivos: {len(ARCHIVOS)} | {total_mb:.0f} MB")
print(f"  (consolidado te_all_raw_v4.parquet: {os.path.getsize(RAW_PATH)/1e6:.0f} MB)")

for cloud, v in by_cloud.items():
    avg_mb = v['mb'] / v['n']
    for f in ARCHIVOS:
        if re.match(rf'te_{cloud.lower()}_', os.path.basename(f)):
            mb = os.path.getsize(f) / 1e6
            if mb < avg_mb * 0.3:
                print(f"  ⚠️  {os.path.basename(f)}: {mb:.0f} MB << avg {avg_mb:.0f} MB — revisar extracción")


INVENTARIO DE ARCHIVOS RAW — 72 parquets

POR PROVEEDOR CLOUD:
  Cloud    Archivos        MB   MB/arch
  ------------------------------------
  AWS            24      1302      54.2
  AZURE          24      1000      41.7
  GCP            24      1004      41.8

POR ÉPOCA TEMPORAL:
  Época      Archivos        MB
  ------------------------------
  apr               3       170
  aug              15       673
  jul               3       153
  jun-jul          24      1078
  may              15       657
  sep              12       576

TOTAL  archivos: 72 | 3306 MB
  (consolidado te_all_raw_v4.parquet: 3306 MB)


In [4]:

print("=" * 70)
print("VALIDACIÓN ESTRUCTURAL — CONTEO POR MÉTRICA (72 parquets)")
print("=" * 70)

metric_counts = {}
null_net = 0
total_net = 0
null_bgp = 0
total_bgp = 0

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'source_agent'])
    for m in METRICS_NET + METRICS_BGP:
        mask = pc.equal(tbl.column('metric'), m)
        sub  = tbl.filter(mask)
        n    = len(sub)
        metric_counts[m] = metric_counts.get(m, 0) + n
        if m == 'network.latency':
            total_net += n
            null_net  += pc.sum(pc.is_null(sub.column('source_agent'))).as_py() or 0
        if m == 'bgp.reachability':
            total_bgp += n
            null_bgp  += pc.sum(pc.is_null(sub.column('source_agent'))).as_py() or 0
    del tbl, sub, mask; gc.collect()

print("\nMÉTRICAS DE RED (agent-to-agent):")
for m in METRICS_NET:
    print(f"  {m:<35} : {metric_counts.get(m,0):>15,}")
total_r = sum(metric_counts.get(m, 0) for m in METRICS_NET)
print(f"  {'--- Total filas RED':<35} : {total_r:>15,}")
print(f"  source_agent NULL en latency       : {null_net/max(total_net,1)*100:>7.1f}%")

print("\nMÉTRICAS BGP (monitores → prefijos IP):")
for m in METRICS_BGP:
    print(f"  {m:<35} : {metric_counts.get(m,0):>15,}")
total_b = sum(metric_counts.get(m, 0) for m in METRICS_BGP)
print(f"  {'--- Total filas BGP':<35} : {total_b:>15,}")
print(f"  source_agent NULL en bgp.reach     : {null_bgp/max(total_bgp,1)*100:>7.1f}%  (100% = diseño correcto)")

ratio = (metric_counts.get('network.latency', 1) /
         max(metric_counts.get('bgp.reachability', 1), 1))
print(f"\nRatio RED/BGP (latency/reachability) : {ratio:.1f}x  (≈7x esperado)")


VALIDACIÓN ESTRUCTURAL — CONTEO POR MÉTRICA (72 parquets)

MÉTRICAS DE RED (agent-to-agent):
  network.latency                     :     141,493,661
  network.loss                        :     142,560,827
  network.jitter                      :     141,488,306
  --- Total filas RED                 :     425,542,794
  source_agent NULL en latency       :     0.4%

MÉTRICAS BGP (monitores → prefijos IP):
  bgp.path_changes.count              :      33,159,080
  bgp.updates.count                   :      33,159,049
  bgp.reachability                    :      33,159,043
  --- Total filas BGP                 :      99,477,172
  source_agent NULL en bgp.reach     :   100.0%  (100% = diseño correcto)

Ratio RED/BGP (latency/reachability) : 4.3x  (≈7x esperado)


In [5]:

from datetime import date

print("=" * 70)
print("COBERTURA TEMPORAL DIARIA — DATASET EXTENDIDO v4")
print("=" * 70)

daily_counts = {}

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['_time'])
    times = tbl.column('_time').cast('timestamp[us]')
    ts    = pd.to_datetime(times.to_pylist(), utc=True)
    for d, cnt in ts.normalize().value_counts().items():
        key = d.date()
        daily_counts[key] = daily_counts.get(key, 0) + cnt
    del tbl, times, ts; gc.collect()

daily = (pd.DataFrame(list(daily_counts.items()), columns=['date','n_rows'])
           .sort_values('date').reset_index(drop=True))

print(f"\nPeríodo          : {daily['date'].min()} → {daily['date'].max()}")
print(f"Días con datos   : {len(daily)}")

dates_sorted = sorted(daily_counts.keys())
gaps = []
for i in range(1, len(dates_sorted)):
    delta = (dates_sorted[i] - dates_sorted[i-1]).days
    if delta > 1:
        gaps.append((dates_sorted[i-1], dates_sorted[i], delta-1))

if not gaps:
    print("Días sin datos   : 0  ✅ sin gaps")
else:
    print(f"Días sin datos   : {len(gaps)} GAPS ENCONTRADOS ⚠️")
    for g0, g1, n in gaps:
        print(f"  {g0} → {g1}: {n} días sin datos")

avg_rows = daily['n_rows'].mean()
print(f"\nFilas/día media  : {avg_rows:>12,.0f}")
print(f"Filas/día mínimo : {daily['n_rows'].min():>12,}  ({daily.loc[daily['n_rows'].idxmin(),'date']})")
print(f"Filas/día máximo : {daily['n_rows'].max():>12,}  ({daily.loc[daily['n_rows'].idxmax(),'date']})")

apr22_rows = daily_counts.get(date(2026, 4, 22), 0)
apr23_28   = [daily_counts.get(date(2026, 4, d), 0) for d in range(23, 29)]
apr23_28_avg = np.mean(apr23_28) if apr23_28 else 1
pct_apr22 = apr22_rows / max(apr23_28_avg, 1) * 100

print(f"\n--- VERIFICACIÓN APR 22 (LOG-34, D41-3) ---")
print(f"Filas Apr 22     : {apr22_rows:>12,}  ({pct_apr22:.1f}% del promedio Apr 23-28)")
print(f"Promedio Apr23-28: {apr23_28_avg:>12,.0f}")
if pct_apr22 < 40:
    print(f"⚠️  Apr 22 < 40% → notebook 03 etiquetado debe iniciar baseline el 23-abr")
    APR22_DISTORSIONA = True
else:
    print(f"✅ Apr 22 ≥ 40% → puede incluirse en el baseline (Apr 22-28)")
    APR22_DISTORSIONA = False



COBERTURA TEMPORAL DIARIA — DATASET EXTENDIDO v4

Período          : 2026-04-22 → 2026-09-30
Días con datos   : 162
Días sin datos   : 0  ✅ sin gaps

Filas/día media  :    3,240,864
Filas/día mínimo :      961,448  (2026-04-22)
Filas/día máximo :    6,095,172  (2026-09-16)

--- VERIFICACIÓN APR 22 (LOG-34, D41-3) ---
Filas Apr 22     :      961,448  (32.3% del promedio Apr 23-28)
Promedio Apr23-28:    2,979,583
⚠️  Apr 22 < 40% → notebook 03 etiquetado debe iniciar baseline el 23-abr


In [6]:

print("=" * 70)
print("ARQUITECTURA THOUSANDEYES — VALIDACIÓN DATASET EXTENDIDO v4")
print("=" * 70)

net_rows = 0; bgp_rows = 0
agents_set = set()
net_routes = set(); bgp_routes = set()
null_net = 0; null_bgp = 0

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'source_agent', 'cloud_provider', 'region'])

    nt = tbl.filter(pc.starts_with(tbl.column('metric'), 'network.'))
    bt = tbl.filter(pc.starts_with(tbl.column('metric'), 'bgp.'))

    net_rows += len(nt)
    null_net += pc.sum(pc.is_null(nt.column('source_agent'))).as_py() or 0
    agents_set.update(
        [a for a in pc.unique(nt.column('source_agent').drop_null()).to_pylist() if a])

    for row in (nt.group_by(['cloud_provider','region'])
                  .aggregate([('metric','count')])
                  .select(['cloud_provider','region']).to_pylist()):
        net_routes.add((row['cloud_provider'], row['region']))

    bgp_rows += len(bt)
    null_bgp += pc.sum(pc.is_null(bt.column('source_agent'))).as_py() or 0

    for row in (bt.group_by(['cloud_provider','region'])
                  .aggregate([('metric','count')])
                  .select(['cloud_provider','region']).to_pylist()):
        bgp_routes.add((row['cloud_provider'], row['region']))

    del tbl, nt, bt; gc.collect()

print(f"\n1. FILAS POR TIPO DE TEST")
print(f"  RED (A2A — Agent-to-Agent)")
print(f"    Filas               : {net_rows:>15,}")
print(f"    source_agent NULL   : {null_net/max(net_rows,1)*100:>9.1f}%  (agentes identificados)")
print(f"    Rutas (cloud×region): {len(net_routes):>15}")

print(f"\n  BGP (monitor → prefijo IP — plano de control)")
print(f"    Filas               : {bgp_rows:>15,}")
print(f"    source_agent NULL   : {null_bgp/max(bgp_rows,1)*100:>9.1f}%  (NULL by design — monitores privados LATAM)")
print(f"    Rutas (cloud×region): {len(bgp_routes):>15}")

print(f"\n2. AGENTES ÚNICOS (test RED)")
print(f"  Agentes únicos   : {len(agents_set)}")
print(f"  Configurado (TF) : {TE_CONFIG['n_agents_configured']}  (agentsCloudFrom.csv, 12 países)")
if len(agents_set) < TE_CONFIG['n_agents_configured']:
    diff = TE_CONFIG['n_agents_configured'] - len(agents_set)
    print(f"  ⚠️  {diff} agente(s) inactivo(s) en el período extendido")

print(f"\n3. RUTAS BGP vs RUTAS DE RED")
both = net_routes & bgp_routes
print(f"  Destinos con AMBAS métricas      : {len(both)}")
print(f"  Cobertura BGP → rutas de red     : {len(both)/max(len(net_routes),1)*100:.1f}%")

if len(net_routes) == 23 and len(bgp_routes) == 23:
    print(f"\n[OK] Arquitectura validada: NET=agentes (source_agent poblado) | BGP=monitores (NULL by design)")
else:
    print(f"\n⚠️  Rutas RED={len(net_routes)}, BGP={len(bgp_routes)} — revisar dataset")

print(f"\nAgentes activos ({len(agents_set)}):")
for a in sorted(agents_set)[:10]:
    print(f"  {a}")
if len(agents_set) > 10:
    print(f"  ... y {len(agents_set)-10} más")


ARQUITECTURA THOUSANDEYES — VALIDACIÓN DATASET EXTENDIDO v4

1. FILAS POR TIPO DE TEST
  RED (A2A — Agent-to-Agent)
    Filas               :     425,542,794
    source_agent NULL   :       0.4%  (agentes identificados)
    Rutas (cloud×region):              23

  BGP (monitor → prefijo IP — plano de control)
    Filas               :      99,477,172
    source_agent NULL   :     100.0%  (NULL by design — monitores privados LATAM)
    Rutas (cloud×region):              23

2. AGENTES ÚNICOS (test RED)
  Agentes únicos   : 45
  Configurado (TF) : 46  (agentsCloudFrom.csv, 12 países)
  ⚠️  1 agente(s) inactivo(s) en el período extendido

3. RUTAS BGP vs RUTAS DE RED
  Destinos con AMBAS métricas      : 23
  Cobertura BGP → rutas de red     : 100.0%

[OK] Arquitectura validada: NET=agentes (source_agent poblado) | BGP=monitores (NULL by design)

Agentes activos (45):
  Belo Horizonte, Brazil (TIM)
  Belo Horizonte, Brazil (Vivo)
  Bogotá, Colombia
  Brasilia, Brazil
  Brasilia, Brazil (Cl

In [7]:

print("=" * 70)
print("DISTRIBUCIÓN POR CLOUD PROVIDER")
print("=" * 70)

cloud_net = {}
cloud_bgp = {}

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'cloud_provider'])
    df  = tbl.to_pandas()
    del tbl; gc.collect()

    for m in METRICS_NET:
        sub = df[df['metric'] == m]
        for cloud, cnt in sub['cloud_provider'].value_counts().items():
            if cloud not in cloud_net:
                cloud_net[cloud] = {}
            cloud_net[cloud][m] = cloud_net[cloud].get(m, 0) + int(cnt)

    for m in METRICS_BGP:
        sub = df[df['metric'] == m]
        for cloud, cnt in sub['cloud_provider'].value_counts().items():
            if cloud not in cloud_bgp:
                cloud_bgp[cloud] = {}
            cloud_bgp[cloud][m] = cloud_bgp[cloud].get(m, 0) + int(cnt)

    del df; gc.collect()

total_net_all = sum(sum(v.values()) for v in cloud_net.values())
total_bgp_all = sum(sum(v.values()) for v in cloud_bgp.values())

print("\nMÉTRICAS DE RED (latency / loss / jitter):")
print(f"  {'Cloud':<8} {'latency':>12}  {'loss':>12}  {'jitter':>12}  {'Total':>12}  {'%':>6}")
print(f"  {'-'*70}")
for cloud in sorted(cloud_net):
    lat  = cloud_net[cloud].get('network.latency', 0)
    loss = cloud_net[cloud].get('network.loss', 0)
    jit  = cloud_net[cloud].get('network.jitter', 0)
    tot  = lat + loss + jit
    pct  = tot / max(total_net_all, 1) * 100
    print(f"  {cloud:<8} {lat:>12,}  {loss:>12,}  {jit:>12,}  {tot:>12,}  {pct:>6.1f}%")

print("\nMÉTRICAS BGP (path_changes / updates / reachability):")
print(f"  {'Cloud':<8} {'path_changes':>14}  {'updates':>12}  {'reachability':>14}  {'Total':>12}  {'%':>6}")
print(f"  {'-'*80}")
for cloud in sorted(cloud_bgp):
    pc_ = cloud_bgp[cloud].get('bgp.path_changes.count', 0)
    up  = cloud_bgp[cloud].get('bgp.updates.count', 0)
    rch = cloud_bgp[cloud].get('bgp.reachability', 0)
    tot = pc_ + up + rch
    pct = tot / max(total_bgp_all, 1) * 100
    print(f"  {cloud:<8} {pc_:>14,}  {up:>12,}  {rch:>14,}  {tot:>12,}  {pct:>6.1f}%")


DISTRIBUCIÓN POR CLOUD PROVIDER

MÉTRICAS DE RED (latency / loss / jitter):
  Cloud         latency          loss        jitter         Total       %
  ----------------------------------------------------------------------
  AWS        55,276,405    55,733,840    55,273,195   166,283,440    39.1%
  Azure      43,067,270    43,405,249    43,065,661   129,538,180    30.4%
  GCP        43,149,986    43,421,738    43,149,450   129,721,174    30.5%

MÉTRICAS BGP (path_changes / updates / reachability):
  Cloud      path_changes       updates    reachability         Total       %
  --------------------------------------------------------------------------------
  AWS          13,000,525    13,000,519      13,000,525    39,001,569    39.2%
  Azure        10,052,798    10,052,793      10,052,792    30,158,383    30.3%
  GCP          10,105,757    10,105,737      10,105,726    30,317,220    30.5%


In [8]:

print("=" * 70)
print("DISTRIBUCIÓN POR REGIÓN CLOUD (métricas de red)")
print("=" * 70)

region_counts = {}

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'cloud_provider', 'region'])
    lat = tbl.filter(pc.equal(tbl.column('metric'), 'network.latency'))
    df  = lat.select(['cloud_provider','region']).to_pandas()
    for (cloud, region), cnt in df.groupby(['cloud_provider','region']).size().items():
        key = (cloud, region)
        region_counts[key] = region_counts.get(key, 0) + int(cnt)
    del tbl, lat, df; gc.collect()

total = sum(region_counts.values())
avg   = total / max(len(region_counts), 1)

print(f"\nTotal filas network.latency : {total:,}")
print(f"Regiones activas           : {len(region_counts)}")
print(f"Promedio por región        : {avg:,.0f}")
print(f"\n{'Cloud':<6} {'Región':<35} {'Filas':>12}  {'%':>6}  Nota")
print(f"  {'-'*75}")

latam_regions = {'sa-east-1', 'southamerica-east1', 'southamerica-west1',
                 'mexicocentral', 'mx-central-1', 'Santiago-Local-Zone',
                 'brazilsoutheast'}

for (cloud, region), cnt in sorted(region_counts.items(), key=lambda x: (-x[1], x[0][0])):
    pct  = cnt / max(total, 1) * 100
    nota = '← LATAM' if region in latam_regions else ''
    print(f"  {cloud:<6} {region:<35} {cnt:>12,}  {pct:>6.2f}%  {nota}")

counts_list = list(region_counts.values())
cv = np.std(counts_list) / np.mean(counts_list) * 100 if counts_list else 0
print(f"\nCoeficiente de variación entre regiones: {cv:.1f}%")
if cv < 10:
    print("  ✅ Distribución uniforme — sin sesgo de región")
else:
    print(f"  ⚠️  CV={cv:.1f}% > 10% — verificar distribución por región")


DISTRIBUCIÓN POR REGIÓN CLOUD (métricas de red)

Total filas network.latency : 141,493,661
Regiones activas           : 23
Promedio por región        : 6,151,898

Cloud  Región                                     Filas       %  Nota
  ---------------------------------------------------------------------------
  GCP    southamerica-east1                     6,174,511    4.36%  ← LATAM
  GCP    us-west2                               6,169,418    4.36%  
  GCP    europe-west3                           6,168,918    4.36%  
  GCP    us-east4                               6,168,302    4.36%  
  Azure  centralus                              6,167,888    4.36%  
  GCP    us-central1                            6,167,554    4.36%  
  Azure  westus                                 6,166,577    4.36%  
  Azure  brazilsoutheast                        6,165,739    4.36%  ← LATAM
  GCP    southamerica-west1                     6,165,708    4.36%  ← LATAM
  Azure  germanywestcentral                    

In [9]:

print("=" * 70)
print("DISTRIBUCIÓN POR CIUDAD LATAM (source_loc)")
print("=" * 70)

city_counts  = {}
city_agents  = {}

for f in ARCHIVOS:
    tbl  = pq.read_table(f, columns=['metric', 'source_agent', 'source_loc'])
    net  = tbl.filter(pc.equal(tbl.column('metric'), 'network.latency'))
    df   = net.select(['source_agent','source_loc']).to_pandas().dropna()
    for loc, grp in df.groupby('source_loc'):
        city_counts[loc]  = city_counts.get(loc, 0) + len(grp)
        city_agents[loc]  = city_agents.get(loc, set()) | set(grp['source_agent'])
    del tbl, net, df; gc.collect()

total_city = sum(city_counts.values())
print(f"\nCiudades LATAM activas: {len(city_counts)}")
print(f"Total filas network.latency: {total_city:,}")
print(f"\n{'#':>3}  {'Ciudad (source_loc)':<45} {'Agentes':>8}  {'Filas':>10}  {'%':>6}")
print(f"  {'-'*80}")

for rank, (loc, cnt) in enumerate(
        sorted(city_counts.items(), key=lambda x: -x[1]), 1):
    n_agents = len(city_agents[loc])
    pct      = cnt / max(total_city, 1) * 100
    isp_flag = ' (2 ISPs)' if n_agents >= 2 else ''
    print(f"  {rank:>3}  {loc:<45} {n_agents:>8}  {cnt:>10,}  {pct:>6.2f}%{isp_flag}")
    if rank == 5:
        remaining = len(city_counts) - 5
        print(f"  ...  ({remaining} ciudades más)")
        break

print(f"\nCiudades con ≥ 2 agentes/ISPs: "
      + str([loc for loc, ag in city_agents.items() if len(ag) >= 2]))


DISTRIBUCIÓN POR CIUDAD LATAM (source_loc)

Ciudades LATAM activas: 27
Total filas network.latency: 140,896,311

  #  Ciudad (source_loc)                            Agentes       Filas       %
  --------------------------------------------------------------------------------
    1  São Paulo, São Paulo, Brazil                         4  12,567,850    8.92% (2 ISPs)
    2  Mexico City, Mexico City, Mexico                     3   9,459,604    6.71% (2 ISPs)
    3  Lima, Peru                                           3   9,430,866    6.69% (2 ISPs)
    4  Santiago, Chile                                      2   6,322,453    4.49% (2 ISPs)
    5  Buenos Aires, Argentina                              2   6,311,713    4.48% (2 ISPs)
  ...  (22 ciudades más)

Ciudades con ≥ 2 agentes/ISPs: ['Belo Horizonte, Minas Gerais, Brazil', 'Brasília, Federal District, Brazil', 'Buenos Aires, Argentina', 'Fortaleza, Ceará, Brazil', 'Lima, Peru', 'Manaus, Amazonas, Brazil', 'Medellín, Colombia', 'Mexico C

In [10]:

from collections import defaultdict

print("=" * 70)
print("MÉTRICAS DE RED POR AGENTE (source_agent)")
print("=" * 70)

agent_metrics = defaultdict(lambda: defaultdict(int))

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'source_agent'])
    df  = tbl.filter(
        pc.and_(pc.starts_with(tbl.column('metric'), 'network.'),
                pc.invert(pc.is_null(tbl.column('source_agent'))))
    ).to_pandas()
    for _, row in df.groupby(['source_agent', 'metric']).size().reset_index(
            name='n').iterrows():
        agent_metrics[row['source_agent']][row['metric']] += int(row['n'])
    del tbl, df; gc.collect()

print(f"\nAgentes con métricas de red: {len(agent_metrics)}")
print(f"\n{'Agente':<50} {'latency':>10}  {'loss':>10}  {'jitter':>10}  OK?")
print(f"  {'-'*95}")

problemas = []
for agent in sorted(agent_metrics.keys()):
    lat  = agent_metrics[agent].get('network.latency', 0)
    loss = agent_metrics[agent].get('network.loss',    0)
    jit  = agent_metrics[agent].get('network.jitter',  0)
    counts = [c for c in [lat, loss, jit] if c > 0]
    if len(counts) == 3:
        cv = (max(counts) - min(counts)) / np.mean(counts) * 100
        ok = '✅' if cv < 5 else '⚠️'
        if cv >= 5:
            problemas.append((agent, lat, loss, jit))
    else:
        ok = '❌' if len(counts) < 3 else '✅'
        if len(counts) < 3:
            problemas.append((agent, lat, loss, jit))
    print(f"  {agent:<50} {lat:>10,}  {loss:>10,}  {jit:>10,}  {ok}")

if not problemas:
    print(f"\n✅ Todos los agentes reportan las 3 métricas de red de forma consistente")
else:
    print(f"\n⚠️  {len(problemas)} agente(s) con cobertura asimétrica:")
    for ag, lat, loss, jit in problemas:
        print(f"  {ag}: lat={lat:,} loss={loss:,} jit={jit:,}")


MÉTRICAS DE RED POR AGENTE (source_agent)

Agentes con métricas de red: 45

Agente                                                latency        loss      jitter  OK?
  -----------------------------------------------------------------------------------------------
  Belo Horizonte, Brazil (TIM)                        3,133,182   3,149,990   3,133,135  ✅
  Belo Horizonte, Brazil (Vivo)                       3,106,625   3,150,404   3,106,600  ✅
  Bogotá, Colombia                                    3,168,186   3,173,434   3,168,152  ✅
  Brasilia, Brazil                                    3,151,786   3,158,280   3,151,714  ✅
  Brasilia, Brazil (Claro)                            3,077,104   3,147,619   3,077,102  ✅
  Buenos Aires, Argentina                             3,163,477   3,174,691   3,162,640  ✅
  Buenos Aires, Argentina (Claro)                     3,148,236   3,162,531   3,148,270  ✅
  Cancun, Mexico (Telmex)                             3,112,208   3,160,146   3,112,156  ✅
  Carac

In [11]:

from collections import defaultdict

print("=" * 70)
print("MÉTRICAS BGP POR DESTINO (cloud_provider × region)")
print("=" * 70)

bgp_dest_metrics = defaultdict(lambda: defaultdict(int))

for f in ARCHIVOS:
    tbl = pq.read_table(f, columns=['metric', 'cloud_provider', 'region'])
    df  = tbl.filter(pc.starts_with(tbl.column('metric'), 'bgp.')).to_pandas()
    for _, row in df.groupby(['cloud_provider', 'region', 'metric']).size().reset_index(
            name='n').iterrows():
        dest = (row['cloud_provider'], row['region'])
        bgp_dest_metrics[dest][row['metric']] += int(row['n'])
    del tbl, df; gc.collect()

print(f"\nDestinos con cobertura BGP: {len(bgp_dest_metrics)}")
print(f"\n{'Cloud':<6} {'Región':<35} {'path_changes':>14}  {'updates':>12}  {'reachab.':>12}  OK?")
print(f"  {'-'*85}")

problemas_bgp = []
for (cloud, region) in sorted(bgp_dest_metrics.keys()):
    d   = bgp_dest_metrics[(cloud, region)]
    pc_ = d.get('bgp.path_changes.count', 0)
    up  = d.get('bgp.updates.count',      0)
    rch = d.get('bgp.reachability',       0)
    counts = [c for c in [pc_, up, rch] if c > 0]
    if len(counts) == 3 and max(counts) - min(counts) <= max(counts) * 0.01:
        ok = '✅'
    else:
        ok = '⚠️'
        problemas_bgp.append((cloud, region, pc_, up, rch))
    print(f"  {cloud:<6} {region:<35} {pc_:>14,}  {up:>12,}  {rch:>12,}  {ok}")

if not problemas_bgp:
    print(f"\n✅ Las 3 métricas BGP tienen volúmenes idénticos en todos los destinos")
    print("   (cada evento BGP genera exactamente 3 filas: path_changes + updates + reachability)")
else:
    print(f"\n⚠️  {len(problemas_bgp)} destino(s) con asimetría BGP:")
    for c, r, p, u, rch in problemas_bgp:
        print(f"  {c}/{r}: pc={p:,} up={u:,} rch={rch:,}")

total_bgp_d = sum(sum(v.values()) for v in bgp_dest_metrics.values())
print(f"\nTotal observaciones BGP : {total_bgp_d:,}")
print(f"Nota: BGP event-driven (RFC 4271 §4) → volumen ~7x menor que métricas de red")
print(f"Monitores BGP activos: 12 privados LATAM (BUE×2 · RIO×2 · GRU×4 · SCL×2 · CDMX×2)")


MÉTRICAS BGP POR DESTINO (cloud_provider × region)

Destinos con cobertura BGP: 23

Cloud  Región                                path_changes       updates      reachab.  OK?
  -------------------------------------------------------------------------------------
  AWS    Chicago-Local-Zone                       1,446,970     1,446,976     1,446,966  ✅
  AWS    Dallas-Local-Zone                        1,444,632     1,444,631     1,444,626  ✅
  AWS    Santiago-Local-Zone                      1,442,202     1,442,198     1,442,192  ✅
  AWS    ap-east-1                                1,439,273     1,439,263     1,439,273  ✅
  AWS    eu-central-1                             1,442,894     1,442,905     1,442,905  ✅
  AWS    mx-central-1                             1,451,514     1,451,514     1,451,508  ✅
  AWS    sa-east-1                                1,437,712     1,437,711     1,437,709  ✅
  AWS    us-east-1                                1,445,060     1,445,058     1,445,068  ✅
  AWS    

In [12]:

print("=" * 70)
print("COBERTURA POR RUTA (source_loc × cloud × región)")
print("=" * 70)

all_daily = []
for f in ARCHIVOS:
    tbl  = pq.read_table(f, columns=['_time','metric','source_loc','cloud_provider','region'])
    lat  = tbl.filter(pc.equal(tbl.column('metric'), 'network.latency'))
    chunk = lat.select(['_time','source_loc','cloud_provider','region']).to_pandas()
    chunk['_time'] = pd.to_datetime(chunk['_time'], utc=True)
    chunk['date']  = chunk['_time'].dt.normalize()
    agg = (chunk.groupby(['source_loc','cloud_provider','region','date'])
                .size().reset_index(name='n'))
    all_daily.append(agg)
    del tbl, lat, chunk, agg; gc.collect()

daily_routes = pd.concat(all_daily, ignore_index=True)
del all_daily; gc.collect()

daily_routes = daily_routes.groupby(
    ['source_loc','cloud_provider','region','date'], as_index=False)['n'].sum()

total_days  = daily_routes['date'].nunique()
route_cover = (daily_routes.groupby(['source_loc','cloud_provider','region'])['date']
                            .nunique()
                            .reset_index(name='days_covered'))
route_cover['coverage_pct'] = route_cover['days_covered'] / total_days * 100
del daily_routes; gc.collect()

n_routes   = len(route_cover)
n_100      = (route_cover['coverage_pct'] >= 99.9).sum()
n_ge90     = (route_cover['coverage_pct'] >= 90).sum()
n_lt80     = (route_cover['coverage_pct'] < 80).sum()
avg_cov    = route_cover['coverage_pct'].mean()
min_cov    = route_cover['coverage_pct'].min()

print(f"\nPeríodo total    : {total_days} días")
print(f"Rutas analizadas : {n_routes}")
print(f"Rutas ≥ 99.9%    : {n_100}")
print(f"Rutas ≥ 90%      : {n_ge90}")
print(f"Rutas < 80%      : {n_lt80}  {'✅' if n_lt80 == 0 else '⚠️'}")
print(f"Cobertura promedio: {avg_cov:.1f}%")
print(f"Cobertura mínima  : {min_cov:.1f}%")

if n_lt80 > 0:
    print(f"\n⚠️  Rutas con cobertura < 80% (requieren análisis antes de etiquetado):")
    low = route_cover[route_cover['coverage_pct'] < 80].sort_values('coverage_pct')
    for _, r in low.iterrows():
        print(f"  {r['source_loc']:<40} → {r['cloud_provider']}/{r['region']:<30}"
              f" {r['days_covered']:>4}d / {total_days}d = {r['coverage_pct']:.1f}%")
else:
    print(f"\n✅ Todas las rutas ≥ 90% de cobertura — no se requiere imputación ni exclusión")


COBERTURA POR RUTA (source_loc × cloud × región)

Período total    : 162 días
Rutas analizadas : 621
Rutas ≥ 99.9%    : 621
Rutas ≥ 90%      : 621
Rutas < 80%      : 0  ✅
Cobertura promedio: 100.0%
Cobertura mínima  : 100.0%

✅ Todas las rutas ≥ 90% de cobertura — no se requiere imputación ni exclusión


In [3]:

print("=" * 70)
print("CADENCIA DE MUESTREO: RED vs BGP — PERÍODO EXTENDIDO")
print("=" * 70)

SAMPLE_ROUTES_NET = [
    ('Lima',          'AWS',   'us-east-1'),
    ('Buenos Aires',  'Azure', 'eastus'),
    ('São Paulo',     'GCP',   'southamerica-east1'),
]

recent_files = []
for _prov in ['aws', 'azure', 'gcp']:
    _matching = [f for f in ARCHIVOS
                 if _prov in f.lower() and any(e in f for e in ['sep_w', 'aug_w'])]
    recent_files.extend(_matching[:2])
if not recent_files:
    recent_files = ARCHIVOS[-6:]

print(f"\nArchivos muestreados: {len(recent_files)}")
for rf in recent_files:
    print(f"  {rf.split('/')[-1]}")

print(f"\nA. RED (A2A) — Terraform A2A_interval = {TE_CONFIG['interval_s']} s")

for (city_key, cloud, region) in SAMPLE_ROUTES_NET:
    diffs_s = []
    for f in recent_files:
        tbl = pq.read_table(f, columns=['_time','metric','source_loc','cloud_provider','region'])
        mask = pc.and_(pc.and_(
            pc.equal(tbl.column('metric'),         'network.latency'),
            pc.equal(tbl.column('cloud_provider'),  cloud)),
            pc.equal(tbl.column('region'),          region))
        sub = tbl.filter(mask).to_pandas()
        sub = sub[sub['source_loc'].str.contains(city_key, na=False)]
        if len(sub) > 1:
            ts = sorted(pd.to_datetime(sub['_time'], utc=True))
            diffs_s.extend([(ts[i+1]-ts[i]).total_seconds() for i in range(min(len(ts)-1, 200))])
        del tbl, sub; gc.collect()
    if diffs_s:
        p50 = np.percentile(diffs_s, 50)
        p95 = np.percentile(diffs_s, 95)
        print(f"  {city_key:<20} → {cloud}/{region:<20}  P50={p50:.0f}s  P95={p95:.0f}s")
    else:
        print(f"  {city_key:<20} → {cloud}/{region:<20}  sin datos (verificar source_loc en C8)")

print(f"\nB. BGP — EVENT-DRIVEN (RFC 4271 §4.4)")
for cloud, region in [('AWS','us-east-1'), ('Azure','mexicocentral'), ('GCP','southamerica-east1')]:
    diffs_s = []
    for f in recent_files:
        tbl  = pq.read_table(f, columns=['_time','metric','cloud_provider','region'])
        mask = pc.and_(pc.and_(
            pc.equal(tbl.column('metric'),        'bgp.path_changes.count'),
            pc.equal(tbl.column('cloud_provider'), cloud)),
            pc.equal(tbl.column('region'),         region))
        sub  = tbl.filter(mask)
        if len(sub) > 1:
            ts = sorted(pd.to_datetime(sub.column('_time').to_pylist(), utc=True))
            diffs_s.extend([(ts[i+1]-ts[i]).total_seconds() for i in range(min(len(ts)-1,300))])
        del tbl, sub; gc.collect()
    if diffs_s:
        p50 = np.percentile(diffs_s, 50)
        p95 = np.percentile(diffs_s, 95)
        p99 = np.percentile(diffs_s, 99)
        print(f"  {cloud}/{region:<22}  P50={p50:.1f}s ({p50/60:.2f}min)  P95={p95:.0f}s ({p95/60:.1f}min)  P99={p99:.0f}s ({p99/60:.1f}min)")
    else:
        print(f"  {cloud}/{region:<22}  sin datos en archivos muestreados")

print(f"\n  BGP desalineación (pipeline 30-min): si P95 << 1800s → forward-fill seguro")
print(f"  BGP desalineación (pipeline 5-min):  si P99 >> 300s  → NaN estructural ~38.8%")

CADENCIA DE MUESTREO: RED vs BGP — PERÍODO EXTENDIDO

Archivos muestreados: 6
  te_aws_raw_aug_w1.parquet
  te_aws_raw_aug_w2.parquet
  te_azure_raw_aug_w1.parquet
  te_azure_raw_aug_w2.parquet
  te_gcp_raw_aug_w1.parquet
  te_gcp_raw_aug_w2.parquet

A. RED (A2A) — Terraform A2A_interval = 300 s
  Lima                 → AWS/us-east-1             P50=0s  P95=288s
  Buenos Aires         → Azure/eastus                P50=0s  P95=288s
  São Paulo            → GCP/southamerica-east1    P50=0s  P95=285s

B. BGP — EVENT-DRIVEN (RFC 4271 §4.4)
  AWS/us-east-1               P50=0.0s (0.00min)  P95=125s (2.1min)  P99=441s (7.4min)
  Azure/mexicocentral           P50=0.0s (0.00min)  P95=125s (2.1min)  P99=441s (7.4min)
  GCP/southamerica-east1      P50=0.0s (0.00min)  P95=125s (2.1min)  P99=441s (7.4min)

  BGP desalineación (pipeline 30-min): si P95 << 1800s → forward-fill seguro
  BGP desalineación (pipeline 5-min):  si P99 >> 300s  → NaN estructural ~38.8%


In [14]:

print("=" * 70)
print("DISTRIBUCIÓN DE VALORES POR MÉTRICA — CUANTILES EXACTOS (DuckDB D11)")
print("=" * 70)

parquet_list = ", ".join(f"'{f}'" for f in ARCHIVOS)

con = duckdb.connect(database=':memory:')
con.execute("SET memory_limit = '4GB'")
con.execute(f"SET temp_directory = '{SAVE_DIR}/duckdb_tmp'")
os.makedirs(f'{SAVE_DIR}/duckdb_tmp', exist_ok=True)

print("\nEjecutando PERCENTILE_CONT sobre todos los parquets vía DuckDB...")
print("(Esto puede tomar 5-10 minutos — DuckDB procesa out-of-core)\n")

query = f'''
SELECT
    metric,
    COUNT(*)                                         AS n,
    PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY value::DOUBLE)  AS p1,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY value::DOUBLE)  AS p50,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY value::DOUBLE)  AS p95,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY value::DOUBLE)  AS p99,
    MAX(value::DOUBLE)                               AS vmax,
    AVG(CASE WHEN value::DOUBLE = 0.0 THEN 1 ELSE 0 END) * 100  AS pct_zero
FROM read_parquet([{parquet_list}])
WHERE metric IN (
    'network.latency', 'network.loss', 'network.jitter',
    'bgp.path_changes.count', 'bgp.updates.count', 'bgp.reachability'
)
GROUP BY metric
ORDER BY metric
'''

result = con.execute(query).df()
con.close()

print(f"{'Métrica':<35} {'N':>15}  {'P1':>8}  {'P50':>8}  {'P95':>8}  {'P99':>8}  {'Max':>10}  {'%=0':>6}")
print(f"  {'-'*105}")
for _, row in result.iterrows():
    print(f"  {row['metric']:<35} {int(row['n']):>15,}  {row['p1']:>8.4f}  {row['p50']:>8.4f}  "
          f"{row['p95']:>8.4f}  {row['p99']:>8.4f}  {row['vmax']:>10.3f}  {row['pct_zero']:>6.1f}%")

print("\nGUARDRAILS ITU-T:")
lat_row = result[result['metric'] == 'network.latency'].iloc[0]
los_row = result[result['metric'] == 'network.loss'].iloc[0]
jit_row = result[result['metric'] == 'network.jitter'].iloc[0]

lat_p95 = lat_row['p95']
print(f"  network.latency P95 = {lat_p95:.4f}s  vs  0.250s (ITU-T Y.1541 Clase 1)  "
      + ("✅" if lat_p95 < 0.250 else "⚠️ EXCEDE"))
loss_p99 = los_row['p99']
print(f"  network.loss    P99 = {loss_p99:.1f}%   vs  3% recomendado (ITU-T G.114)   "
      + ("✅" if loss_p99 < 3 else "⚠️"))
jit_p99 = jit_row['p99']
print(f"  network.jitter  P99 = {jit_p99:.1f}ms  vs  50ms límite (ITU-T G.114)       "
      + ("✅" if jit_p99 < 50 else "⚠️"))
print(f"\n(Resultados exactos — DuckDB PERCENTILE_CONT, sin muestreo)")

import shutil
shutil.rmtree(f'{SAVE_DIR}/duckdb_tmp', ignore_errors=True)


DISTRIBUCIÓN DE VALORES POR MÉTRICA — CUANTILES EXACTOS (DuckDB D11)

Ejecutando PERCENTILE_CONT sobre todos los parquets vía DuckDB...
(Esto puede tomar 5-10 minutos — DuckDB procesa out-of-core)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Métrica                                           N        P1       P50       P95       P99         Max     %=0
  ---------------------------------------------------------------------------------------------------------
  bgp.path_changes.count                   33,159,080    0.0000    1.0000    3.0000    6.0000      29.000    21.1%
  bgp.reachability                         33,159,043    0.0000  100.0000  100.0000  100.0000     100.000     3.5%
  bgp.updates.count                        33,159,049    1.0000    2.0000    6.0000   11.0000      62.000     0.0%
  network.jitter                          141,488,306    0.0000    0.2245    4.0952    7.9796    2143.000    13.6%
  network.latency                         141,493,661    0.0021    0.0830    0.2211    0.3335      13.565     0.2%
  network.loss                            142,560,827    0.0000    0.0000    6.0000   24.0000     100.000    82.0%

GUARDRAILS ITU-T:
  network.latency P95 = 0.2211s  vs  0.250s (ITU-T Y.1541 Clase 1)  ✅
 

In [15]:

print("=" * 70)
print("BGP MULTIPLICIDAD 1:45 — DATASET EXTENDIDO (72 parquets, 161 días)")
print("=" * 70)

all_agg = []
for f in ARCHIVOS:
    tbl  = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric'])
    lat  = tbl.filter(pc.equal(tbl.column('metric'), 'network.latency'))
    chunk = lat.to_pandas()
    chunk['_time']  = pd.to_datetime(chunk['_time'], utc=True)
    chunk['bucket'] = chunk['_time'].dt.floor('30min')
    agg = (chunk.groupby(['cloud_provider','region','bucket'])['source_agent']
                .nunique().reset_index(name='n_agents'))
    all_agg.append(agg)
    del tbl, lat, chunk, agg; gc.collect()

apd = pd.concat(all_agg, ignore_index=True)
del all_agg; gc.collect()

print(f"\nVentanas 30-min × destino analizadas: {len(apd):,}")
print(f"Período: {apd['bucket'].min()} → {apd['bucket'].max()}")

stats = apd['n_agents'].describe(percentiles=[.50,.90,.95,.99])
print(f"\nAgentes A2A compartiendo 1 señal BGP por (cloud × región × ventana 30-min):")
print(f"  count: {stats['count']:.0f}  |  mean: {stats['mean']:.1f}  |  "
      f"std: {stats['std']:.1f}  |  min: {stats['min']:.0f}  |  max: {stats['max']:.0f}")
print(f"  50%: {stats['50%']:.0f}  |  90%: {stats['90%']:.0f}  |  "
      f"95%: {stats['95%']:.0f}  |  99%: {stats['99%']:.0f}")

print("\nDistribución de n_agents:")
for val, cnt in apd['n_agents'].value_counts().sort_index().items():
    pct  = cnt / len(apd) * 100
    bars = '█' * int(pct / 2)
    print(f"  {val:>3} agentes: {cnt:>8,} casos ({pct:>5.1f}%) {bars}")

moda = apd['n_agents'].mode()[0]
print(f"\nModa: {moda} agentes comparten el valor BGP de un destino")
print(f"→ 1 señal BGP : {moda} rutas A2A — multiplicidad confirmada sobre {apd['bucket'].dt.date.nunique()} días")
print(f"→ Raíz causal #1 de H1 EDA RECHAZADA (consistente con dataset original v3)")

del apd; gc.collect()


BGP MULTIPLICIDAD 1:45 — DATASET EXTENDIDO (72 parquets, 161 días)

Ventanas 30-min × destino analizadas: 178,048
Período: 2026-04-22 15:30:00+00:00 → 2026-09-30 23:30:00+00:00

Agentes A2A compartiendo 1 señal BGP por (cloud × región × ventana 30-min):
  count: 178048  |  mean: 44.9  |  std: 0.3  |  min: 40  |  max: 45
  50%: 45  |  90%: 45  |  95%: 45  |  99%: 45

Distribución de n_agents:
   40 agentes:        1 casos (  0.0%) 
   41 agentes:        1 casos (  0.0%) 
   42 agentes:        3 casos (  0.0%) 
   43 agentes:      493 casos (  0.3%) 
   44 agentes:   17,638 casos (  9.9%) ████
   45 agentes:  159,912 casos ( 89.8%) ████████████████████████████████████████████

Moda: 45 agentes comparten el valor BGP de un destino
→ 1 señal BGP : 45 rutas A2A — multiplicidad confirmada sobre 162 días
→ Raíz causal #1 de H1 EDA RECHAZADA (consistente con dataset original v3)


0

In [4]:

print("=" * 70)
print("DESALINEACIÓN TEMPORAL BGP — DATASET EXTENDIDO")
print("=" * 70)

bgp_targets = [('AWS','us-east-1'), ('Azure','mexicocentral'), ('GCP','southamerica-east1')]
sample_files = ARCHIVOS[::5][:15]

print(f"\nArchivos muestreados: {len(sample_files)} (distribuidos a lo largo del período)")
print(f"Tests A2A   : intervalo={TE_CONFIG['interval_s']}s fijo · DSCP {TE_CONFIG['dscp']} · plano de datos")
print(f"BGP monitors: ciclo variable · plano de control · independiente de A2A")

for cloud, region in bgp_targets:
    all_diffs = []
    for f in sample_files:
        tbl  = pq.read_table(f, columns=['_time','metric','cloud_provider','region'])
        mask = pc.and_(pc.and_(
            pc.equal(tbl.column('metric'),        'bgp.path_changes.count'),
            pc.equal(tbl.column('cloud_provider'), cloud)),
            pc.equal(tbl.column('region'),         region))
        sub  = tbl.filter(mask)
        if len(sub) > 1:
            ts   = sorted(pd.to_datetime(sub.column('_time').to_pylist(), utc=True))
            diff = [(ts[i+1]-ts[i]).total_seconds() for i in range(min(len(ts)-1, 500))]
            all_diffs.extend(diff)
        del tbl, sub; gc.collect()

    if all_diffs:
        arr  = np.array(all_diffs)
        p50  = np.percentile(arr, 50)
        p95  = np.percentile(arr, 95)
        p99  = np.percentile(arr, 99)
        win30 = 1800
        win5  = 300
        ff30  = '✅ forward-fill seguro' if p95 < win30 else '⚠️ revisar'
        ff5   = f'⚠️ NaN estructural (P99={p99:.0f}s >> {win5}s)' if p99 > win5 else '✅'
        print(f"\n  {cloud}/{region}  (N_gaps={len(all_diffs):,})")
        print(f"    P50={p50:.0f}s ({p50/60:.2f}min)  P95={p95:.0f}s ({p95/60:.1f}min)  P99={p99:.0f}s ({p99/60:.1f}min)")
        print(f"    Pipeline 30-min (ventana=1800s): P95={p95:.0f}s << 1800s  {ff30}")
        print(f"    Pipeline 5-min  (ventana=300s):  P99={p99:.0f}s  {'>> 300s  '+ff5 if p99>win5 else '<< 300s  ✅'}")

print(f"\nNota: DSCP EF (datos) ≠ BGP (routing) — señales físicamente distintas.")
print(f"  Validación empírica de NaN rate en features BGP → notebook 04b (análisis FE)")

DESALINEACIÓN TEMPORAL BGP — DATASET EXTENDIDO

Archivos muestreados: 15 (distribuidos a lo largo del período)
Tests A2A   : intervalo=300s fijo · DSCP EF (46) · plano de datos
BGP monitors: ciclo variable · plano de control · independiente de A2A

  AWS/us-east-1  (N_gaps=2,500)
    P50=0s (0.00min)  P95=127s (2.1min)  P99=762s (12.7min)
    Pipeline 30-min (ventana=1800s): P95=127s << 1800s  ✅ forward-fill seguro
    Pipeline 5-min  (ventana=300s):  P99=762s  >> 300s  ⚠️ NaN estructural (P99=762s >> 300s)

  Azure/mexicocentral  (N_gaps=2,500)
    P50=0s (0.00min)  P95=85s (1.4min)  P99=632s (10.5min)
    Pipeline 30-min (ventana=1800s): P95=85s << 1800s  ✅ forward-fill seguro
    Pipeline 5-min  (ventana=300s):  P99=632s  >> 300s  ⚠️ NaN estructural (P99=632s >> 300s)

  GCP/southamerica-east1  (N_gaps=2,500)
    P50=0s (0.00min)  P95=89s (1.5min)  P99=434s (7.2min)
    Pipeline 30-min (ventana=1800s): P95=89s << 1800s  ✅ forward-fill seguro
    Pipeline 5-min  (ventana=300s):  P99=

In [5]:

def _ts_utc(t):
    ts = pd.Timestamp(t)
    return ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')

PRE_WIN  = pd.Timedelta('60min')
POST_WIN = pd.Timedelta('30min')
N_SAMPLE = 30
RNG      = np.random.default_rng(42)

DEST_SAMPLE = [
    ('AWS',   'us-east-1'),
    ('AWS',   'sa-east-1'),
    ('Azure', 'mexicocentral'),
    ('Azure', 'eastus'),
    ('GCP',   'southamerica-east1'),
    ('GCP',   'us-east4'),
]

print("=" * 70)
print("CORRELACIÓN BGP path_changes → network.latency POR DESTINO")
print(f"  Ventana pre: 60min | Ventana post: 30min | N por destino: {N_SAMPLE}")
print("=" * 70)
print(f"\n{'Cloud':<6} {'Región':<35} {'N':>5}  {'Ratio':>7}  Interpretación")
print(f"  {'-'*70}")

results = []
for (cloud, region) in DEST_SAMPLE:
    bgp_events_d = []
    lat_pairs    = []

    for f in ARCHIVOS:
        tbl = pq.read_table(f, columns=['_time','metric','value','cloud_provider','region'])
        mask_dest = pc.and_(
            pc.equal(tbl.column('cloud_provider'), cloud),
            pc.equal(tbl.column('region'), region))
        sub = tbl.filter(mask_dest)
        if len(sub) == 0:
            del tbl, sub; gc.collect(); continue

        df = sub.to_pandas()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)

        bgp = df[(df['metric']=='bgp.path_changes.count') &
                 (pd.to_numeric(df['value'], errors='coerce').fillna(0) > 0)]
        bgp_events_d.extend(bgp['_time'].tolist())

        lat = df[df['metric']=='network.latency'][['_time','value']]
        lat_pairs.extend(zip(lat['_time'].tolist(),
                             pd.to_numeric(lat['value'], errors='coerce').tolist()))
        del tbl, sub, df, bgp, lat; gc.collect()

    if not bgp_events_d or not lat_pairs:
        print(f"  {cloud:<6} {region:<35}  sin datos suficientes")
        continue

    lat_ts = pd.Series({t: v for t, v in lat_pairs if pd.notna(v)}).sort_index()
    ev_sample = RNG.choice(bgp_events_d,
                           size=min(N_SAMPLE, len(bgp_events_d)), replace=False)
    route_ratios = []
    for t in ev_sample:
        t = _ts_utc(t)
        pre_v  = lat_ts[(lat_ts.index >= t - PRE_WIN)  & (lat_ts.index < t)].median()
        post_v = lat_ts[(lat_ts.index >  t)            & (lat_ts.index <= t + POST_WIN)].median()
        if pd.notna(pre_v) and pd.notna(post_v) and pre_v > 0:
            route_ratios.append(post_v / pre_v)

    del lat_ts, lat_pairs, bgp_events_d; gc.collect()

    if not route_ratios:
        print(f"  {cloud:<6} {region:<35}  ventanas sin solapamiento")
        continue

    ratio  = np.median(route_ratios)
    interp = 'sin efecto' if abs(ratio-1.0)<0.05 else ('↑ post-BGP' if ratio>1.0 else '↓ post-BGP')
    results.append(ratio)
    print(f"  {cloud:<6} {region:<35} {len(route_ratios):>5}  {ratio:>7.3f}x  {interp}")

if results:
    med = np.median(results)
    print(f"\n  GLOBAL mediana ratio = {med:.3f}x")
    if abs(med-1.0) < 0.05:
        print("  ✅ Ratio ≈ 1.000x — confirma hallazgo v9: BGP es feature de CONTEXTO, no causal")
    else:
        print(f"  ⚠️  Ratio {med:.3f}x — revisar hallazgo para dataset extendido")
    print(f"\n  → En LATAM, path_changes reflejan ingeniería de tráfico activa, no convergencia post-falla.")
    print(f"  → bgp.path_changes.count = feature de CONTEXTO, no predictor causal de latencia.")

CORRELACIÓN BGP path_changes → network.latency POR DESTINO
  Ventana pre: 60min | Ventana post: 30min | N por destino: 30

Cloud  Región                                  N    Ratio  Interpretación
  ----------------------------------------------------------------------
  AWS    us-east-1                              30    1.001x  sin efecto
  AWS    sa-east-1                              30    1.007x  sin efecto
  Azure  mexicocentral                          30    0.998x  sin efecto
  Azure  eastus                                 30    1.003x  sin efecto
  GCP    southamerica-east1                     30    0.983x  sin efecto
  GCP    us-east4                               30    1.002x  sin efecto

  GLOBAL mediana ratio = 1.001x
  ✅ Ratio ≈ 1.000x — confirma hallazgo v9: BGP es feature de CONTEXTO, no causal

  → En LATAM, path_changes reflejan ingeniería de tráfico activa, no convergencia post-falla.
  → bgp.path_changes.count = feature de CONTEXTO, no predictor causal de latencia.
